# Long Short-Term Memory (LSTM)

![LSTM](./images/LTSM.png)

## Concept Overview
**Long Short-Term Memory (LSTM)** is a type of **Recurrent Neural Network (RNN)** designed to handle the **vanishing gradient problem** in long sequence learning.  
It introduces a **cell state** (long-term memory) and **gates** (controllers for information flow) that decide what to keep, update, or forget at each timestep.

**Key Idea:**  
Instead of only passing a hidden state through time, LSTM maintains a **cell state** that can carry information across many timesteps with minimal changes.

---

## Mathematics

At time step t:

1. **Forget Gate**  

$$ f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) $$ 
Decides how much of the previous cell state to keep.

2. **Input Gate**  
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$  
Controls how much new information to store.

3. **Candidate Cell State**  
$$
\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)
$$  
Potential new memory values.

4. **Cell State Update**  
$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$  
Combines old memory with the new candidate values.

5. **Output Gate**  
$$
o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)
$$  
Decides what part of the cell state to output.

6. **Hidden State Update**  
$$
h_t = o_t \odot \tanh(c_t)
$$  
Produces the output for this timestep.

---

### Summary
Together, these gates and updates illustrate how the LSTM carefully balances **retaining useful memory**, **adding new information**, and **deciding what to output**, all on a per-unit basis. This selective control is why LSTMs work well for sequences with long-range dependencies.



## Applications
- Machine Translation
- Speech Recognition
- Text Generation
- Time-Series Forecasting
- Music Composition

---

##  Advantages
- Handles long-term dependencies better than vanilla RNNs.
- Mitigates vanishing/exploding gradient problems.
- Flexible in remembering or forgetting information.

---

##  Limitations
- Computationally expensive compared to vanilla RNN.
- More parameters → higher risk of overfitting on small datasets.
- Still struggles with very long sequences (Transformers often outperform for such cases).

---

## LSTM Cell
![LSTM](./images/LTSM.png)

## LSTM Computation Graph:

![LSTM Computation Graph](./images/lstm_graph.png)

## Manual Computation Of Gates values

In [7]:
import torch
import torch.nn as nn

# Parameters
input_size = 10    # features per timestep
hidden_size = 20   # hidden state size
num_layers = 2     # stacked LSTM layers
seq_len = 5        # timesteps
batch_size = 3

# Model
lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

# Input: (batch, seq_len, input_size)
x = torch.randn(batch_size, seq_len, input_size)

# Forward pass
output, (h_n, c_n) = lstm(x)

print("Output shape:", output.shape)  # (batch, seq_len, hidden_size)
print("Hidden state shape:", h_n.shape)  # (num_layers, batch, hidden_size)
print("Cell state shape:", c_n.shape)    # (num_layers, batch, hidden_size)

Output shape: torch.Size([3, 5, 20])
Hidden state shape: torch.Size([2, 3, 20])
Cell state shape: torch.Size([2, 3, 20])


## From Scratch Implementation:

Forget gate =>  
$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)
$$

Input gate =>  
$$
i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)
$$

Candidate cell state =>  
$$
\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)
$$

Cell state update =>  
$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

Output gate =>  
$$
o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)
$$

Hidden state update =>  
$$
h_t = o_t \odot \tanh(c_t)
$$


![](./images/LTSM.png)

In [6]:
import torch
import torch.nn as nn

class LSTMCellCustom(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat((h_prev, x), dim=1)
        f_t = torch.sigmoid(self.W_f(combined))
        i_t = torch.sigmoid(self.W_i(combined))
        g_t = torch.tanh(self.W_c(combined))
        c_t = f_t * c_prev + i_t * g_t
        o_t = torch.sigmoid(self.W_o(combined))
        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t

# Example usage
input_size = 10
hidden_size = 20
lstm_cell = LSTMCellCustom(input_size, hidden_size)

x_t = torch.randn(3, input_size)
h_t = torch.zeros(3, hidden_size)
c_t = torch.zeros(3, hidden_size)

h_next, c_next = lstm_cell(x_t, h_t, c_t)


## Step-by-Step Numerical Example

**Given:**
- $ x_t = [0.5, -0.2, 0.1, 0.7] $
- $ h_{t-1} = [0.1, 0.4, -0.3] $
- $ c_{t-1} = [0.2, -0.5, 0.3] $

(Weights and biases initialized randomly for demonstration.)

**Computed Results:**

1. Forget gate:  

$$ f_t \approx [0.5512, 0.5742, 0.5278] $$

2. Input gate:  
$$
i_t \approx [0.4621, 0.5976, 0.5243]
$$

3. Candidate cell state:  
$$
\tilde{c}_t \approx [-0.0937, 0.4043, -0.2274]
$$

4. New cell state:  
$$
c_t \approx [0.0670, -0.0455, 0.0391]
$$

5. Output gate:  
$$
o_t \approx [0.5730, 0.4620, 0.4873]
$$

6. New hidden state:  
$$
h_t \approx [0.0383, -0.0210, 0.0190]
$$

---

## Interpretation of Computed Results

1. **Forget gate: $$ f_t \approx [0.5512, 0.5742, 0.5278] $$**  
Each value here is between 0 and 1 and controls how much of the **previous cell state** to keep for each hidden unit.  
- Values around 0.55 mean the model **retains about 55%** of the previous memory for each corresponding unit,  
- It "forgets" or discards the remaining 45%.  
This selective forgetting helps the LSTM drop irrelevant or outdated information.

2. **Input gate: $$ i_t \approx [0.4621, 0.5976, 0.5243] $$**  
These values control how much **new information** (candidate cell state) gets added to the cell state.  
- For the first unit, about 46% of the candidate info will be accepted,  
- For the second, ~60%,  
- And for the third, ~52%.  
This allows the model to update its memory selectively based on the new input and context.

3. **Candidate cell state: $$ \tilde{c}_t \approx [-0.0937, 0.4043, -0.2274] $$**  
These are the **proposed new values** (ranging roughly from -1 to 1) to add to the cell memory, before gating.  
- Positive values (like 0.4043) indicate an increase in that feature’s memory,  
- Negative values (like -0.0937 and -0.2274) indicate decreasing or suppressing that feature.  
The LSTM uses this to create a nuanced update to its internal memory.

4. **New cell state: $$ c_t \approx [0.0670, -0.0455, 0.0391] $$**  
This is the **updated memory** after combining the forgotten old memory and added new candidate info.  
- Notice the values are closer to zero than before, indicating the model may be focusing on very refined or weak memory signals at this step.  
- This balancing act allows the network to keep useful info while discarding noise.

5. **Output gate: $$ o_t \approx [0.5730, 0.4620, 0.4873] $$**  
This gate controls how much of the **cell state** to reveal as the new hidden state $$ h_t $$.  
- The values between ~0.46 to 0.57 indicate partial exposure of the internal memory.  
- This gating mechanism lets the LSTM regulate what information to pass on to the next layer or time step.

6. **New hidden state: $$ h_t \approx [0.0383, -0.0210, 0.0190] $$**  
This is the **final output** of the LSTM cell at this timestep, which will be fed forward or to the next timestep.  
- The small magnitude suggests the network outputs a subtle signal here, potentially indicating careful, incremental updates.  
- Hidden state values combine the cell state filtered by the output gate and a non-linearity (tanh).